# AI / Private Credit / Insurance Stress Dashboard

A monitoring notebook for the thesis:

> **AI / data-center economics weaken → structured/private credit deteriorates → insurer capital pressure rises → funding stress appears → policy response follows.**

The dashboard deliberately separates **observable evidence** from the broader thesis. A deterioration in one layer does **not** imply insurer insolvency.

## Monitoring layers

1. **Compute economics**
2. **Credit markets**
3. **Private credit**
4. **Insurance balance sheets**
5. **Insurance funding / liquidity**
6. **Policy response**

### Public sources used automatically

- Federal Reserve / FRED, including Financial Accounts (Z.1)
- SEC EDGAR XBRL and earnings releases (collected by `scripts/fetch_*.py`):
  BDC credit metrics, life-insurer GAAP balance sheets, AI capex and revenue
- Vast.ai public GPU marketplace prices

### File-driven sources

These do not have a dependable free real-time API, so the notebook accepts CSV inputs:

- Data-center ABS / structured-credit rating actions
- NAIC statutory insurer metrics (optional: overrides the SEC GAAP proxy)
- Bermuda reinsurance / BSCR metrics
- Hand-entered BDC values for any field SEC filings leave blank

The notebook will create empty templates for those files if they do not exist.

In [ ]:

# Optional installs:
# %pip install pandas numpy matplotlib requests

from pathlib import Path
from io import StringIO
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_DIR = Path("stress_dashboard_data")
DATA_DIR.mkdir(exist_ok=True)

START_DATE = "2018-01-01"
print(f"Data directory: {DATA_DIR.resolve()}")



## 1. Configuration

FRED's public graph CSV endpoint does not require an API key. The core automatic series are:

| Series | Meaning | Layer |
|---|---|---|
| `DGS10` | 10-year Treasury yield | Credit baseline |
| `BAMLC0A0CM` | ICE BofA US Corporate OAS | Credit |
| `BAMLH0A0HYM2` | ICE BofA US High Yield OAS | Credit |
| `WALCL` | Federal Reserve total assets | Policy response |
| `WPC` | Primary credit, weekly average | Policy / liquidity |
| `BOGZ1FL543190543Q` | Life insurers' funding agreements backing securities | Insurance funding |

You can add more FRED series to `FRED_SERIES`.


In [ ]:

FRED_SERIES = {
    "DGS10": "10Y Treasury Yield",
    "BAMLC0A0CM": "IG Corporate OAS",
    "BAMLH0A0HYM2": "HY Corporate OAS",
    "WALCL": "Fed Total Assets",
    "WPC": "Fed Primary Credit",
    "BOGZ1FL543190543Q": "Life Insurer FABS Liabilities",
}

def fetch_fred_csv(series_id: str, start: str = START_DATE) -> pd.Series:
    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={series_id}&cosd={start}"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df.set_index("date")[series_id]

def load_fred_panel(series_map=FRED_SERIES, start=START_DATE):
    series = {}
    errors = {}
    for sid, label in series_map.items():
        try:
            series[label] = fetch_fred_csv(sid, start)
        except Exception as e:
            errors[sid] = str(e)

    if not series:
        raise RuntimeError(f"No FRED series could be loaded. Errors: {errors}")

    panel = pd.concat(series, axis=1).sort_index()
    return panel, errors

fred, fred_errors = load_fred_panel()
print("Latest FRED observations:")
display(fred.ffill().tail(5))

if fred_errors:
    print("Series that failed:")
    display(pd.Series(fred_errors, name="error"))



## 2. Derived market indicators

The goal is not a black-box score. These are transparent transformations:

- **IG / HY OAS z-scores:** current spread relative to trailing 3-year history.
- **FABS growth:** year-over-year growth in life-insurer funding-agreement liabilities.
- **Primary-credit stress:** current borrowing relative to its own trailing distribution.
- **Fed balance-sheet impulse:** 13-week percentage change.

Higher values generally indicate more stress, except Fed balance-sheet growth, which is interpreted as a **policy-response** indicator rather than fundamental deterioration.


In [ ]:
def rolling_zscore(s: pd.Series, window=756, min_periods=126):
    # Rolling observations, not calendar days. Avoid forward-filled weekends.
    mu = s.rolling(window, min_periods=min_periods).mean()
    sigma = s.rolling(window, min_periods=min_periods).std()
    return (s - mu) / sigma.replace(0, np.nan)

derived = pd.DataFrame(index=fred.index)
for col in ["IG Corporate OAS", "HY Corporate OAS"]:
    if col in fred:
        derived[f"{col} z"] = rolling_zscore(fred[col].dropna()).reindex(fred.index)

if "Fed Primary Credit" in fred:
    # Weekly observations; 156 weeks = ~3 years.
    derived["Fed Primary Credit z"] = rolling_zscore(
        np.log1p(fred["Fed Primary Credit"].dropna().clip(lower=0)),
        window=156, min_periods=52,
    ).reindex(fred.index)

if "Life Insurer FABS Liabilities" in fred:
    # Quarterly series: compare four observations, not 365 forward-filled days.
    derived["FABS YoY %"] = fred["Life Insurer FABS Liabilities"].dropna().pct_change(4, fill_method=None).mul(100).reindex(fred.index)

if "Fed Total Assets" in fred:
    derived["Fed Assets 13w %"] = fred["Fed Total Assets"].dropna().pct_change(13, fill_method=None).mul(100).reindex(fred.index)

# A last-known value is usable only when it is fresh for that source's frequency.
MAX_AGE_DAYS = {
    "10Y Treasury Yield": 7, "IG Corporate OAS": 7,
    "HY Corporate OAS": 7, "Fed Total Assets": 14,
    "Fed Primary Credit": 14, "Life Insurer FABS Liabilities": 180,
}
# FRED dates quarterly observations at the quarter's start; the data describe the
# quarter's end. Z.1 publishes ~10 weeks after quarter end, so measure age from there.
PERIOD_END = {"Life Insurer FABS Liabilities": pd.offsets.QuarterEnd(0)}
as_of = pd.Timestamp.now(tz="UTC").tz_localize(None).normalize()
freshness = []
for label in FRED_SERIES.values():
    s = fred[label].dropna() if label in fred else pd.Series(dtype=float)
    last = s.index.max() if len(s) else pd.NaT
    period_end = last + PERIOD_END[label] if label in PERIOD_END and pd.notna(last) else last
    age = (as_of - period_end).days if pd.notna(period_end) else np.nan
    freshness.append({"series": label, "last_observed": last, "period_end": period_end, "age_days": age,
                      "max_age_days": MAX_AGE_DAYS[label],
                      "fresh": bool(pd.notna(age) and 0 <= age <= MAX_AGE_DAYS[label])})
freshness_df = pd.DataFrame(freshness)
display(freshness_df)
display(derived.tail(10))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11))
for label in ["IG Corporate OAS", "HY Corporate OAS"]:
    if label in fred:
        fred[label].dropna().plot(ax=axes[0], label=label)
axes[0].set_title("Public Credit Spreads")
axes[0].legend()
for label, axis, title in [
    ("Life Insurer FABS Liabilities", axes[1], "Life Insurer FABS Liabilities"),
    ("Fed Primary Credit", axes[2], "Federal Reserve Primary Credit"),
]:
    if label in fred:
        fred[label].dropna().plot(ax=axis)
    axis.set_title(title)
plt.tight_layout()
plt.show()

## 2b. Compute economics

Collected by `scripts/fetch_compute.py`:

- **AI capex:** trailing-twelve-month capital expenditure of Microsoft, Alphabet, Amazon, Meta and Oracle (SEC XBRL). A sharp deceleration would be the first sign that AI investment economics are weakening.
- **Supplier revenue:** NVIDIA TTM revenue growth; CoreWeave capex and revenue as the leveraged neocloud.
- **GPU rental prices:** median on-demand $/GPU-hour on the Vast.ai marketplace. History accumulates across runs; the trend is scored once it spans 60+ days.

In [ ]:
compute_sec_path = DATA_DIR / "compute_sec.csv"
gpu_path = DATA_DIR / "gpu_price_history.csv"
compute_sec = pd.read_csv(compute_sec_path, parse_dates=["date"]) if compute_sec_path.exists() else pd.DataFrame()
gpu_prices = pd.read_csv(gpu_path) if gpu_path.exists() else pd.DataFrame()
compute_signals = {}

if not compute_sec.empty:
    for c in ["quarter_usd_m", "ttm_usd_m", "ttm_yoy_pct"]:
        compute_sec[c] = pd.to_numeric(compute_sec[c], errors="coerce")
    recent = compute_sec[(as_of - compute_sec["date"]).dt.days.between(0, 150)]
    capex = (recent[(recent.metric == "capex") & (recent.group == "hyperscaler")]
             .dropna(subset=["ttm_usd_m", "ttm_yoy_pct"]).sort_values("date").groupby("company").tail(1))
    if len(capex) >= 3:
        year_ago = capex["ttm_usd_m"] / (1 + capex["ttm_yoy_pct"] / 100)
        compute_signals["Hyperscaler capex TTM YoY %"] = 100 * (capex["ttm_usd_m"].sum() / year_ago.sum() - 1)
    nvda = recent[(recent.company == "NVIDIA") & (recent.metric == "revenue")].dropna(subset=["ttm_yoy_pct"])
    if len(nvda):
        compute_signals["NVIDIA revenue TTM YoY %"] = float(nvda.sort_values("date")["ttm_yoy_pct"].iloc[-1])
    display(compute_sec.sort_values("date").groupby(["company", "metric"]).tail(1)
            [["date", "company", "metric", "quarter_usd_m", "ttm_usd_m", "ttm_yoy_pct", "source_url"]])
else:
    print("Compute SEC data: NO DATA (run scripts/fetch_compute.py with SEC_USER_AGENT)")

if not gpu_prices.empty:
    gpu_prices["observed_at"] = pd.to_datetime(gpu_prices["observed_at"], utc=True).dt.tz_localize(None)
    h100 = gpu_prices[gpu_prices.gpu == "H100 SXM"].sort_values("observed_at")
    span = (gpu_prices.observed_at.max() - gpu_prices.observed_at.min()).days
    if len(h100) and (h100.observed_at.max() - h100.observed_at.min()).days >= 60:
        latest = h100.iloc[-1]
        base = h100[h100.observed_at <= latest.observed_at - pd.Timedelta(days=90)]
        base = base.iloc[-1] if len(base) else h100.iloc[0]
        compute_signals["H100 rental price change %"] = 100 * (latest.median_usd_per_gpu_hour / base.median_usd_per_gpu_hour - 1)
    display(gpu_prices.sort_values("observed_at").groupby("gpu").tail(1))
    print(f"GPU price history spans {span} days; the price trend is scored once it spans 60+ days.")

display(pd.Series(compute_signals, name="value", dtype=float).round(2))

In [ ]:
if not compute_sec.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for company, g in compute_sec[compute_sec.metric == "capex"].groupby("company"):
        axes[0].plot(g["date"], g["quarter_usd_m"] / 1000, marker="o", label=company)
    axes[0].set_title("Quarterly capex, $B")
    axes[0].legend(fontsize=8)
    for company, g in compute_sec[compute_sec.metric == "capex"].dropna(subset=["ttm_yoy_pct"]).groupby("company"):
        axes[1].plot(g["date"], g["ttm_yoy_pct"], marker="o", label=company)
    axes[1].axhline(10, linestyle="--", linewidth=1, color="grey")
    axes[1].set_title("Capex TTM YoY % (stress below 10%)")
    for company, g in compute_sec[compute_sec.metric == "revenue"].dropna(subset=["ttm_yoy_pct"]).groupby("company"):
        axes[2].plot(g["date"], g["ttm_yoy_pct"], marker="o", label=company)
    axes[2].axhline(15, linestyle="--", linewidth=1, color="grey")
    axes[2].set_title("Revenue TTM YoY %")
    axes[2].legend(fontsize=8)
    for ax in axes:
        ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()


## 3. Structured-credit / data-center rating-action feed

Populate `rating_actions.csv` with rating actions from S&P, Moody's, Fitch, KBRA, Morningstar DBRS, etc.

Suggested focus:

- Data-center ABS / CMBS / securitizations
- Digital-infrastructure debt
- CLOs containing meaningful AI/data-center borrower exposure
- Senior tranches owned by insurers
- Negative outlook / watch changes even before a downgrade

The dashboard weights **senior-tranche downgrades** more heavily than junior-tranche actions because the thesis depends on deterioration reaching insurer-owned high-grade paper.


In [ ]:

rating_path = DATA_DIR / "rating_actions.csv"

rating_columns = [
    "date", "issuer", "deal", "asset_type", "agency",
    "tranche", "old_rating", "new_rating", "action",
    "seniority", "notional_usd_m", "ai_data_center_exposure_pct",
    "source_url", "notes",
]

if not rating_path.exists():
    pd.DataFrame(columns=rating_columns).to_csv(rating_path, index=False)

ratings = pd.read_csv(rating_path)
if len(ratings):
    ratings["date"] = pd.to_datetime(ratings["date"])
    ratings["notional_usd_m"] = pd.to_numeric(ratings["notional_usd_m"], errors="coerce")
    ratings["ai_data_center_exposure_pct"] = pd.to_numeric(
        ratings["ai_data_center_exposure_pct"], errors="coerce"
    )

display(ratings.tail(20))
print(f"Edit/import: {rating_path.resolve()}")


In [ ]:

ACTION_POINTS = {
    "upgrade": -1.0,
    "positive outlook": -0.5,
    "affirm": 0.0,
    "negative outlook": 1.0,
    "watch negative": 1.5,
    "downgrade": 2.0,
    "default": 4.0,
}

SENIORITY_MULTIPLIER = {
    "senior": 1.5,
    "mezzanine": 1.0,
    "junior": 0.7,
}

def rating_stress_score(df):
    if df.empty:
        return pd.DataFrame(columns=["date", "rating_stress"])
    x = df.copy()
    x["base"] = x["action"].str.lower().map(ACTION_POINTS).fillna(0)
    x["seniority_mult"] = x["seniority"].str.lower().map(SENIORITY_MULTIPLIER).fillna(1)
    exposure = x["ai_data_center_exposure_pct"].fillna(50).clip(0, 100) / 100
    notional = np.log1p(x["notional_usd_m"].fillna(100).clip(lower=0)) / np.log1p(1000)
    x["points"] = x["base"] * x["seniority_mult"] * (0.5 + exposure) * notional
    return (
        x.set_index("date")["points"]
        .resample("30D").sum()
        .rename("rating_stress")
        .to_frame()
    )

rating_stress = rating_stress_score(ratings)
display(rating_stress.tail(12))


## 4. Private-credit / BDC feed

For a first pass, track a basket of large BDCs / private-credit managers.

The strongest deterioration signals are usually:

- **Non-accruals % of portfolio**
- **PIK income / total investment income**
- **NAV change**
- **Interest coverage**
- **Realized credit losses**

PIK is useful because reported income can remain high while borrowers stop paying cash interest.

`scripts/fetch_bdc_sec.py` fills `bdc_metrics.csv` from SEC filings: NAV, PIK share, realized losses and interest coverage from XBRL, and non-accruals (at amortized cost and at fair value) parsed from each earnings release with the quoted sentence saved in `bdc_non_accrual_evidence.csv`. `non_accrual_pct` is at amortized cost, the more conservative basis, because fair value already marks troubled loans down. `realized_loss_pct` is the quarter's net realized loss as % of the portfolio at fair value.

In [ ]:
bdc_path = DATA_DIR / "bdc_metrics.csv"

bdc_columns = [
    "date", "ticker", "manager",
    "non_accrual_pct", "non_accrual_fv_pct", "pik_income_pct",
    "nav_per_share", "nav_qoq_pct",
    "realized_loss_pct", "interest_coverage",
    "source_url", "notes",
]

if not bdc_path.exists():
    pd.DataFrame(columns=bdc_columns).to_csv(bdc_path, index=False)

bdc = pd.read_csv(bdc_path)
if len(bdc):
    bdc["date"] = pd.to_datetime(bdc["date"])
    for c in [
        "non_accrual_pct", "non_accrual_fv_pct", "pik_income_pct", "nav_per_share",
        "nav_qoq_pct", "realized_loss_pct", "interest_coverage"
    ]:
        bdc[c] = pd.to_numeric(bdc[c], errors="coerce")

display(bdc.tail(20))
print(f"Edit/import: {bdc_path.resolve()}")

In [ ]:
# SEC-derived inputs: coverage and the quoted evidence behind each non-accrual figure.
for name in ["bdc_sec_coverage.csv", "bdc_non_accrual_evidence.csv"]:
    p = DATA_DIR / name
    if p.exists():
        display(pd.read_csv(p).tail(12))
    else:
        print(f"{name}: NO DATA (run scripts/fetch_bdc_sec.py with SEC_USER_AGENT)")

if len(bdc):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    panels = [
        ("non_accrual_pct", "Non-accruals, % of portfolio at cost"),
        ("pik_income_pct", "PIK interest, % of investment income"),
        ("nav_per_share", "NAV per share, indexed to first quarter = 100"),
        ("interest_coverage", "Interest coverage, x"),
    ]
    for ax, (col, title) in zip(axes.flat, panels):
        for ticker, g in bdc.dropna(subset=[col]).sort_values("date").groupby("ticker"):
            y = g[col] / g[col].iloc[0] * 100 if col == "nav_per_share" else g[col]
            ax.plot(g["date"], y, marker="o", label=ticker)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=30)
    axes[0, 0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Insurer balance-sheet feed

Populate `insurer_statutory.csv` from NAIC statutory filings / insurer statements when available.

Without statutory data, `scripts/fetch_insurers_sec.py` writes a **GAAP proxy** (`insurer_sec_gaap.csv`) for Athene, Corebridge, F&G, Jackson, Lincoln and Brighthouse: GAAP equity in place of capital & surplus, gross unrealized losses on AFS debt, recurring Level 3 assets from the XBRL instance, and the RBC ratio quoted from the earnings release (`insurer_rbc_evidence.csv`). GAAP equity is not statutory surplus, so the proxy is scored on changes and loss coverage rather than the statutory thresholds.

The most useful normalized measures are against **capital & surplus**, because a $5B loss means very different things at insurers of different size.

Priority fields:

- RBC ratio
- Level 3 assets / capital & surplus
- Schedule BA / capital & surplus
- CLO + ABS / capital & surplus
- Unrealized losses / capital & surplus
- Reinsurance recoverables / capital & surplus
- Ceded reserves / total reserves
- Affiliated investments / capital & surplus
- NAIC-1 → lower designation migration

In [ ]:
insurer_path = DATA_DIR / "insurer_statutory.csv"

insurer_columns = [
    "date", "insurer", "parent",
    "total_assets_usd_m", "capital_surplus_usd_m", "rbc_ratio_pct",
    "level3_assets_usd_m", "schedule_ba_usd_m",
    "clo_abs_usd_m", "unrealized_losses_usd_m",
    "reinsurance_recoverables_usd_m", "ceded_reserves_usd_m",
    "total_reserves_usd_m", "affiliated_investments_usd_m",
    "naic1_to_lower_usd_m", "source_url", "notes",
]

if not insurer_path.exists():
    pd.DataFrame(columns=insurer_columns).to_csv(insurer_path, index=False)

def with_capital_ratios(df):
    df = df.copy()
    if not len(df):
        return df
    df["date"] = pd.to_datetime(df["date"])
    for c in df.columns:
        if c.endswith("_usd_m") or c.endswith("_pct"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    cap = df["capital_surplus_usd_m"].replace(0, np.nan)
    df["level3_to_capital"] = df["level3_assets_usd_m"] / cap
    df["schedule_ba_to_capital"] = df["schedule_ba_usd_m"] / cap
    df["clo_abs_to_capital"] = df["clo_abs_usd_m"] / cap
    df["unrealized_loss_to_capital"] = df["unrealized_losses_usd_m"] / cap
    df["reinsurance_recoverables_to_capital"] = df["reinsurance_recoverables_usd_m"] / cap
    df["affiliated_investments_to_capital"] = df["affiliated_investments_usd_m"] / cap
    df["rating_migration_to_capital"] = df["naic1_to_lower_usd_m"] / cap
    return df

insurers = with_capital_ratios(pd.read_csv(insurer_path))
display(insurers.tail(20))
print(f"Edit/import: {insurer_path.resolve()}")

gaap_path = DATA_DIR / "insurer_sec_gaap.csv"
insurers_gaap = with_capital_ratios(pd.read_csv(gaap_path)) if gaap_path.exists() else pd.DataFrame()
if len(insurers_gaap):
    insurers_gaap["equity_ex_aoci_usd_m"] = insurers_gaap["capital_surplus_usd_m"] - insurers_gaap["aoci_usd_m"].fillna(0)
    display(insurers_gaap.sort_values("date").groupby("insurer").tail(1)[[
        "date", "insurer", "capital_surplus_usd_m", "aoci_usd_m", "equity_ex_aoci_usd_m", "rbc_ratio_pct",
        "level3_to_capital", "unrealized_loss_to_capital", "source_url"]])
    for name in ["insurer_sec_coverage.csv", "insurer_rbc_evidence.csv"]:
        if (DATA_DIR / name).exists():
            display(pd.read_csv(DATA_DIR / name))
else:
    print("Insurer GAAP proxy: NO DATA (run scripts/fetch_insurers_sec.py with SEC_USER_AGENT)")

In [ ]:
if len(insurers_gaap):
    latest_g = insurers_gaap.sort_values("date").groupby("insurer").tail(1).set_index("insurer")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    for name, g in insurers_gaap.sort_values("date").groupby("insurer"):
        axes[0].plot(g["date"], g["equity_ex_aoci_usd_m"] / g["equity_ex_aoci_usd_m"].iloc[0] * 100, marker="o", label=name)
    axes[0].set_title("GAAP equity ex-AOCI, indexed = 100")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].legend(fontsize=7)
    latest_g["unrealized_loss_to_capital"].dropna().sort_values().plot.barh(ax=axes[1])
    axes[1].axvline(1, linestyle="--", linewidth=1, color="grey")
    axes[1].set_title("Gross unrealized losses / GAAP equity")
    latest_g["level3_to_capital"].dropna().sort_values().plot.barh(ax=axes[2])
    axes[2].set_title("Level 3 assets / GAAP equity (context, unscored)")
    for ax in axes[1:]:
        ax.set_ylabel("")
    plt.tight_layout()
    plt.show()


## 6. Bermuda / offshore reinsurance feed

Track the offshore leg separately rather than assuming the U.S. parent's balance sheet captures the risk.

Useful fields:

- Reinsurance assets
- Capital & surplus
- BSCR ratio
- Private credit / total assets
- Structured credit / total assets
- Related-party assets / capital
- U.S.-origin ceded reserves, where available


In [ ]:

bermuda_path = DATA_DIR / "bermuda_reinsurance.csv"

bermuda_columns = [
    "date", "entity",
    "reinsurance_assets_usd_m", "capital_surplus_usd_m",
    "bscr_ratio_pct", "private_credit_pct_assets",
    "structured_credit_pct_assets", "related_party_assets_usd_m",
    "us_origin_ceded_reserves_usd_m", "source_url", "notes",
]

if not bermuda_path.exists():
    pd.DataFrame(columns=bermuda_columns).to_csv(bermuda_path, index=False)

bermuda = pd.read_csv(bermuda_path)
if len(bermuda):
    bermuda["date"] = pd.to_datetime(bermuda["date"])
    for c in bermuda_columns:
        if c.endswith("_usd_m") or c.endswith("_pct"):
            bermuda[c] = pd.to_numeric(bermuda[c], errors="coerce")

display(bermuda.tail(20))
print(f"Edit/import: {bermuda_path.resolve()}")


## 7. Six-layer dashboard

This deliberately uses **status bands**, not a claim of probability of crisis.

- **0–1:** normal / low signal
- **1–2:** watch
- **2–3:** elevated
- **3+:** high stress

For public market variables the score uses rolling z-scores. File-driven components only contribute once you populate them.

- **Compute economics:** mean of `max(0, (10 − hyperscaler capex TTM YoY %) / 10)`, `max(0, (15 − NVIDIA revenue TTM YoY %) / 15)` and, once 60+ days of history exist, `max(0, −H100 rental price change % / 15)`; needs two components.
- **Private credit:** BDC medians (at least three issuers per metric) of non-accruals at cost, PIK share and NAV change.
- **Insurance balance sheets:** statutory formula when `insurer_statutory.csv` is complete; otherwise the GAAP proxy sums `max(0, (400 − median RBC) / 50)`, `max(0, −median YoY change in equity ex-AOCI % / 10)` and `max(0, median unrealized losses / equity − 1)`, needing two components. Equity excludes AOCI so mark-to-market losses are not counted twice.

The `basis` column says which inputs produced each score.

In [ ]:
def latest_valid(s, default=np.nan):
    if s is None or len(s.dropna()) == 0:
        return default
    return float(s.dropna().iloc[-1])

def fresh(label):
    x = freshness_df.set_index("series")
    return label in x.index and bool(x.loc[label, "fresh"])

def positive_z(s):
    z = latest_valid(s)
    return max(0.0, z) if np.isfinite(z) else np.nan

scores = {}
basis = {}
compute_parts = {}
if "Hyperscaler capex TTM YoY %" in compute_signals:
    compute_parts["capex"] = max(0, (10 - compute_signals["Hyperscaler capex TTM YoY %"]) / 10)
if "NVIDIA revenue TTM YoY %" in compute_signals:
    compute_parts["NVIDIA revenue"] = max(0, (15 - compute_signals["NVIDIA revenue TTM YoY %"]) / 15)
if "H100 rental price change %" in compute_signals:
    compute_parts["GPU prices"] = max(0, -compute_signals["H100 rental price change %"] / 15)
scores["1 Compute economics"] = np.mean(list(compute_parts.values())) if len(compute_parts) >= 2 else np.nan
basis["1 Compute economics"] = ", ".join(compute_parts) or "no data"
credit_parts = [
    positive_z(derived[f"{label} z"])
    for label in ["IG Corporate OAS", "HY Corporate OAS"]
    if fresh(label) and f"{label} z" in derived
]
credit_parts = [x for x in credit_parts if np.isfinite(x)]
scores["2 Credit markets"] = np.mean(credit_parts) if len(credit_parts) == 2 else np.nan
basis["2 Credit markets"] = "FRED IG/HY OAS z-scores"

# CSV data require source and age checks before scoring.
def recent_rows(df, entity, max_age=150):
    if df.empty:
        return df
    x = df.copy()
    x["date"] = pd.to_datetime(x["date"], errors="coerce")
    x = x.dropna(subset=["date", "source_url"])
    x = x[x["source_url"].astype(str).str.startswith(("https://", "http://"))]
    x = x[(as_of - x["date"]).dt.days.between(0, max_age)]
    return x.sort_values("date").groupby(entity).tail(1)

latest_bdc = recent_rows(bdc, "ticker")
bdc_metrics = ["non_accrual_pct", "pik_income_pct", "nav_qoq_pct"]
if not latest_bdc.empty and (latest_bdc[bdc_metrics].notna().sum() >= 3).all():
    nonacc, pik, nav = (latest_bdc[c].median() for c in bdc_metrics)
    scores["3 Private credit"] = max(0, (nonacc - 1) / 2) + max(0, (pik - 5) / 10) + max(0, -nav / 5)
    basis["3 Private credit"] = f"median of {len(latest_bdc)} BDCs: non-accrual {nonacc:.1f}%, PIK {pik:.1f}%, NAV QoQ {nav:+.1f}%"
else:
    scores["3 Private credit"] = np.nan
    basis["3 Private credit"] = "fewer than 3 BDCs report every metric"

latest_ins = recent_rows(insurers, "insurer", max_age=450)
required_ins = ["rbc_ratio_pct", "level3_to_capital", "unrealized_loss_to_capital", "rating_migration_to_capital"]
if not latest_ins.empty and latest_ins[required_ins].notna().all(axis=None):
    rbc, level3, unreal, migration = (latest_ins[c].median() for c in required_ins)
    scores["4 Insurance balance sheets"] = max(0, (300-rbc)/100) + max(0, level3-1) + max(0, unreal) + max(0, migration*2)
    basis["4 Insurance balance sheets"] = f"statutory, {len(latest_ins)} insurers"
else:
    scores["4 Insurance balance sheets"] = np.nan
    basis["4 Insurance balance sheets"] = "no data"
    if len(insurers_gaap):
        g = insurers_gaap.dropna(subset=["date"]).sort_values("date")
        latest_g = g[(as_of - g["date"]).dt.days.between(0, 150)].groupby("insurer").tail(1)
        yoy = []
        for _, row in latest_g.iterrows():
            prior = g[(g.insurer == row.insurer) & ((row.date - g.date).dt.days.between(350, 380))]
            if len(prior) and prior.equity_ex_aoci_usd_m.iloc[-1] > 0:
                yoy.append(100 * (row.equity_ex_aoci_usd_m / prior.equity_ex_aoci_usd_m.iloc[-1] - 1))
        parts = {}
        if latest_g.rbc_ratio_pct.notna().sum() >= 2:
            parts["RBC"] = max(0, (400 - latest_g.rbc_ratio_pct.median()) / 50)
        if len(yoy) >= 3:
            parts["equity ex-AOCI YoY"] = max(0, -np.median(yoy) / 10)
        if latest_g.unrealized_loss_to_capital.notna().sum() >= 3:
            parts["unrealized losses / equity"] = max(0, latest_g.unrealized_loss_to_capital.median() - 1)
        if len(parts) >= 2:
            scores["4 Insurance balance sheets"] = sum(parts.values())
            basis["4 Insurance balance sheets"] = f"GAAP proxy, {len(latest_g)} insurers: " + ", ".join(
                f"{k} {v:.2f}" for k, v in parts.items())

funding_parts = []
if fresh("Fed Primary Credit") and "Fed Primary Credit z" in derived:
    funding_parts.append(positive_z(derived["Fed Primary Credit z"]))
if fresh("Life Insurer FABS Liabilities") and "FABS YoY %" in derived:
    fabs_growth = latest_valid(derived["FABS YoY %"])
    funding_parts.append(max(0, (fabs_growth - 10) / 20) if np.isfinite(fabs_growth) else np.nan)
funding_parts = [x for x in funding_parts if np.isfinite(x)]
scores["5 Funding / liquidity"] = np.mean(funding_parts) if funding_parts else np.nan
basis["5 Funding / liquidity"] = "FRED primary credit z-score, Z.1 FABS growth"

policy_parts = []
if fresh("Fed Total Assets") and "Fed Assets 13w %" in derived:
    impulse = latest_valid(derived["Fed Assets 13w %"])
    if np.isfinite(impulse):
        policy_parts.append(max(0, impulse / 5))
if fresh("Fed Primary Credit") and "Fed Primary Credit z" in derived:
    z = positive_z(derived["Fed Primary Credit z"])
    if np.isfinite(z):
        policy_parts.append(z)
scores["6 Policy response"] = np.mean(policy_parts) if policy_parts else np.nan
basis["6 Policy response"] = "FRED Fed balance sheet impulse, primary credit"

score_df = pd.DataFrame({"layer": list(scores), "stress_score": list(scores.values())})
def band(x):
    if pd.isna(x): return "NO DATA"
    if x < 1: return "LOW"
    if x < 2: return "WATCH"
    if x < 3: return "ELEVATED"
    return "HIGH"
score_df["status"] = score_df["stress_score"].map(band)
score_df["basis"] = score_df["layer"].map(basis)
display(score_df)
print(f"Scored layers: {score_df.stress_score.notna().sum()}/6; missing layers are NOT low risk.")

In [ ]:
BAND_COLORS = {"LOW": "#4c9a6a", "WATCH": "#d8a23a", "ELEVATED": "#d9713a", "HIGH": "#c0392b", "NO DATA": "#b0b0b0"}
plot_df = score_df.copy()

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(plot_df["layer"], plot_df["stress_score"].fillna(0), color=plot_df["status"].map(BAND_COLORS))
for y, (score, status) in enumerate(zip(plot_df["stress_score"], plot_df["status"])):
    label = status if pd.isna(score) else f"{status} {score:.2f}"
    ax.text((0 if pd.isna(score) else score) + 0.03, y, label, va="center", fontsize=9)
for x in (1, 2, 3):
    ax.axvline(x, linestyle="--", linewidth=1, color="grey")
ax.set_xlim(0, max(3.5, plot_df["stress_score"].max() + 0.6))
ax.invert_yaxis()
ax.set_title("AI / Private Credit / Insurance Stress Dashboard")
ax.set_xlabel("Stress score (heuristic, not probability)")
plt.tight_layout()
plt.show()


## 8. Thesis confirmation matrix

The dashboard should be interpreted sequentially.

| Observation | Interpretation |
|---|---|
| Compute economics weakens, credit stays fine | AI economics issue; insurer thesis not confirmed |
| Credit spreads widen, but senior ratings stable | Market concern, limited regulatory-capital transmission |
| Senior data-center / structured debt downgraded | Major thesis milestone |
| BDC non-accrual + PIK rise | Private-credit deterioration broadening |
| Insurer RBC falls + rating migration rises | Balance-sheet transmission is occurring |
| FABS / insurer funding stress appears | Liquidity channel activating |
| Fed facilities / balance sheet expand | Policy-response phase |

The strongest evidence would be a **sequence**, not one red indicator.


## 9. Data-source notes

### FRED
The St. Louis Fed provides a REST API and public data downloads. The notebook uses the public graph CSV endpoint for convenience.

### Life-insurer FABS
Federal Reserve Financial Accounts series `BOGZ1FL543190543Q` measures life-insurance-company funding agreements backing securities, quarterly, end-of-period.

### NAIC
NAIC statutory filings provide Schedule D, Schedule BA, Schedule S, Schedule Y and RBC-related information. Beginning with year-end 2026 reporting, additional private-credit / private-rating / PIK-related disclosures should make this layer substantially more useful.

### Bermuda
The Bermuda Monetary Authority publishes long-term insurance market, reinsurance, liquidity, private-credit and stress-test reports. Those releases are currently best treated as periodic report ingestion rather than a high-frequency API.

### SEC EDGAR
`scripts/fetch_bdc_sec.py`, `scripts/fetch_insurers_sec.py` and `scripts/fetch_compute.py` read XBRL companyfacts, XBRL instance documents and earnings-release exhibits (8-K item 2.02). Figures parsed from prose keep the quoted sentence and URL in an `*_evidence.csv` file. Requests carry the `SEC_USER_AGENT` identity required by SEC's fair-access policy.

### Vast.ai
The public marketplace API lists on-demand GPU offers; the median $/GPU-hour is a spot indicator of compute pricing, not a contract price.

## Remaining gaps

1. **Ratings-agency actions** still need manual entry: rating announcements are not published in a free machine-readable feed, and NRSRO XBRL rating histories lag 12 months.
2. **NAIC statutory data** (Schedule D / BA / S, NAIC designation migration) requires a paid feed; the GAAP proxy stands in.
3. **Bermuda** BMA reports are annual PDFs.
4. A **data-center deal registry** mapping borrower → facility → tenant → securitization → insurer holdings.
5. Alert thresholds and historical backtesting.

In [ ]:
snapshot = score_df.copy()
snapshot["as_of"] = pd.Timestamp.utcnow().isoformat()
snapshot_path = DATA_DIR / "latest_stress_snapshot.csv"
snapshot.to_csv(snapshot_path, index=False)
freshness_df.to_csv(DATA_DIR / "source_freshness.csv", index=False)
print(f"Saved snapshot: {snapshot_path.resolve()}")
display(snapshot)